# ConformerFlow — Colab Pro Training

**Before running:**
1. `Runtime → Change runtime type → A100 GPU`
2. Run cells top to bottom
3. Keep this tab open (or use Colab Pro background execution)

**What you need in Google Drive:**
- `MyDrive/conformerflow/ckpt_latest.pt` — checkpoint from Kaggle session2
- `MyDrive/conformerflow/conformerflow_data/` — parsed NMR data (optional, can re-download)

**How to get checkpoint from Kaggle:**
1. Go to `kaggle.com/datasets/perinbamkumar/conformerflow-ckpt-session2`
2. Click Download → extract → find `ckpt_latest.pt`
3. Upload to Google Drive at `MyDrive/conformerflow/ckpt_latest.pt`

In [ ]:
# Cell 1 — Check GPU
import torch
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
print('BF16 support:', torch.cuda.is_bf16_supported())

In [ ]:
# Cell 2 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

# Create conformerflow folder in Drive if it doesn't exist
Path('/content/drive/MyDrive/conformerflow').mkdir(parents=True, exist_ok=True)
print('Drive mounted. conformerflow folder ready.')
print('Contents:', os.listdir('/content/drive/MyDrive/conformerflow'))

In [ ]:
# Cell 3 — Clone repo and install
import shutil, os
from pathlib import Path

if Path('/content/conformerflow').exists():
    shutil.rmtree('/content/conformerflow')

!git clone -b claude/setup-model-training-E3Cdr \
    https://github.com/kumar23kan/conformerflow.git \
    /content/conformerflow

!pip install -q biopython gemmi requests tqdm numpy einops scipy
print('Done')

In [ ]:
# Cell 4 — Set working directory
import os, sys
os.chdir('/content/conformerflow/files')
sys.path.insert(0, '/content/conformerflow/files')
print('Working directory:', os.getcwd())

In [ ]:
# Cell 5 — Load parsed data
# Option A: Copy from Drive (if you already uploaded it)
# Option B: Download from Kaggle using API

import shutil, os, json
from pathlib import Path

# Create directories
for d in ['pdb_data/parsed_nmr', 'pdb_data/splits', 'checkpoints']:
    Path(d).mkdir(parents=True, exist_ok=True)

drive_data = '/content/drive/MyDrive/conformerflow/conformerflow_data'

if Path(drive_data).exists():
    # Option A: Load from Drive
    print('Loading data from Drive...')
    shutil.copytree(f'{drive_data}/parsed_nmr', 'pdb_data/parsed_nmr', dirs_exist_ok=True)
    shutil.copytree(f'{drive_data}/splits',     'pdb_data/splits',     dirs_exist_ok=True)
    print('Data loaded from Drive')
else:
    # Option B: Download from Kaggle API
    print('Drive data not found. Downloading from Kaggle...')
    print('Upload kaggle.json to /content/ first, then re-run this cell.')
    kaggle_json = Path('/content/kaggle.json')
    if kaggle_json.exists():
        Path('/root/.kaggle').mkdir(exist_ok=True)
        shutil.copy(kaggle_json, '/root/.kaggle/kaggle.json')
        os.chmod('/root/.kaggle/kaggle.json', 0o600)
        !kaggle datasets download -d perinbamkumar/conformerflow-data -p /content/kdata --unzip
        shutil.copytree('/content/kdata/parsed_nmr', 'pdb_data/parsed_nmr', dirs_exist_ok=True)
        shutil.copytree('/content/kdata/splits',     'pdb_data/splits',     dirs_exist_ok=True)
        print('Data downloaded from Kaggle')
    else:
        print('ERROR: No data source found.')
        print('Either:')
        print('  1. Upload conformerflow_data/ to Drive at MyDrive/conformerflow/conformerflow_data/')
        print('  2. Upload kaggle.json to /content/kaggle.json')

# Check data
for split in ['train', 'val', 'test']:
    path = f'pdb_data/splits/{split}.json'
    if Path(path).exists():
        data = json.load(open(path))
        print(f'{split}: {len(data)} entries')

In [ ]:
# Cell 6 — Load checkpoint from Drive
import shutil, os
from pathlib import Path

ckpt_sources = [
    '/content/drive/MyDrive/conformerflow/checkpoints/ckpt_latest.pt',
    '/content/drive/MyDrive/conformerflow/ckpt_latest.pt',
    '/content/drive/MyDrive/conformerflow/ckpt_best.pt',
]

Path('checkpoints').mkdir(exist_ok=True)
loaded = False
for src in ckpt_sources:
    if Path(src).exists():
        shutil.copy(src, 'checkpoints/ckpt_resume.pt')
        size = Path(src).stat().st_size / 1e6
        print(f'Checkpoint loaded: {src} ({size:.0f} MB)')
        loaded = True
        break

if not loaded:
    print('ERROR: No checkpoint found in Drive.')
    print('Upload ckpt_latest.pt to MyDrive/conformerflow/')

In [ ]:
# Cell 7 — Train (A100 optimized: bf16, batch_size=32)
# Auto-saves checkpoint to Drive every 2000 steps
!cd /content/conformerflow/files && python3 train.py \
    --config configs/base_config.yaml \
    --batch_size 32 \
    --max_epochs 200 \
    --max_residues 300 \
    --bf16 \
    --resume_from checkpoints/ckpt_resume.pt \
    --drive_backup /content/drive/MyDrive/conformerflow/checkpoints

In [ ]:
# Cell 8 — Save checkpoints to Google Drive
import shutil, os
from pathlib import Path

src = Path('/content/conformerflow/files/checkpoints')
dst = Path('/content/drive/MyDrive/conformerflow/checkpoints')
dst.mkdir(parents=True, exist_ok=True)

if src.exists() and list(src.glob('*.pt')):
    for f in src.glob('*.pt'):
        shutil.copy(f, dst / f.name)
        size = f.stat().st_size / 1e6
        print(f'Saved: {f.name} ({size:.0f} MB)')
    print(f'All checkpoints saved to Drive: {dst}')
else:
    print('No checkpoints found.')